# CIS 5450 — Olist Delivery Delay Prediction
## Part 1: Data Integration, Cleaning & Preprocessing

This notebook prepares an order-level dataset from the Olist Brazilian E-Commerce Dataset for delivery delay prediction.




### 1. Imports and file paths

In [1]:
from pathlib import Path
import pandas as pd

# Works when Jupyter runs from either notebooks/ or the repository root.
if Path('../data/raw').is_dir():
    raw_dir = Path('../data/raw')
elif Path('data/raw').is_dir():
    raw_dir = Path('data/raw')
else:
    raise FileNotFoundError(
        'Cannot find data/raw/. Put the two CSV files under '
        'CIS5450-Olist-Project/data/raw/ and open this notebook from notebooks/ or the repo root.'
    )

orders_path = raw_dir / 'olist_orders_dataset.csv'
items_path = raw_dir / 'olist_order_items_dataset.csv'

for path in (orders_path, items_path):
    if not path.is_file():
        raise FileNotFoundError(f'Missing input CSV: {path}')

print(f'Raw data directory: {raw_dir.resolve()}')

Raw data directory: C:\Users\yym-9\Documents\GitHub\CIS5450_FINAL_PROJECT\data\raw


### 2. Load the raw datasets

In [2]:
orders = pd.read_csv(orders_path)
items = pd.read_csv(items_path)

print(f'Orders shape: {orders.shape}')
print(f'Order items shape: {items.shape}')

Orders shape: (99441, 8)
Order items shape: (112650, 7)


### 3. Preview rows and columns

In [3]:
display(orders.head())
display(items.head())

print('Orders columns:', orders.columns.tolist())
print('Order items columns:', items.columns.tolist())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


Orders columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
Order items columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']


### 4. Data types and missing values

In [4]:
print('Orders dtypes:')
print(orders.dtypes)
print('\nOrder items dtypes:')
print(items.dtypes)

print('\nMissing values in orders:')
display(orders.isna().sum().sort_values(ascending=False).to_frame('missing_count'))

print('Missing values in order items:')
display(items.isna().sum().sort_values(ascending=False).to_frame('missing_count'))

Orders dtypes:
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

Order items dtypes:
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

Missing values in orders:


,missing_count
order_delivered_customer_date,2965
order_delivered_carrier_date,1783
order_approved_at,160
order_id,0
order_purchase_timestamp,0
order_status,0
customer_id,0
order_estimated_delivery_date,0


Missing values in order items:


,missing_count
order_id,0
order_item_id,0
product_id,0
seller_id,0
shipping_limit_date,0
price,0
freight_value,0


### 5. Duplicates and order-level relationships

Before merging, we need to confirm the keys and check how many items and distinct sellers can belong to the same order.

In [5]:
print('Duplicate full rows — orders:', orders.duplicated().sum())
print('Duplicate full rows — order items:', items.duplicated().sum())
print('Duplicate order_id in orders:', orders['order_id'].duplicated().sum())

items_per_order = items.groupby('order_id').size()
sellers_per_order = items.groupby('order_id')['seller_id'].nunique()

print('\nNumber of unique orders in order items:', items['order_id'].nunique())
print('Orders with more than one item:', (items_per_order > 1).sum())
print('Orders with more than one seller:', (sellers_per_order > 1).sum())

print('\nItems per order — summary:')
display(items_per_order.describe().to_frame('items_per_order'))
print('Distinct sellers per order — counts:')
display(sellers_per_order.value_counts().sort_index().to_frame('number_of_orders'))

Duplicate full rows — orders: 0
Duplicate full rows — order items: 0
Duplicate order_id in orders: 0

Number of unique orders in order items: 98666
Orders with more than one item: 9803
Orders with more than one seller: 1278

Items per order — summary:


,items_per_order
count,98666.000000
mean,1.141731
std,0.538452
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,21.000000


Distinct sellers per order — counts:


,number_of_orders
seller_id,
1,97388
2,1219
3,54
4,3
5,2


### 6. Order status overview

In [6]:
display(orders['order_status'].value_counts(dropna=False).to_frame('number_of_orders'))

,number_of_orders
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


### 7. Missing Delivery Dates by Order Status

Check whether missing actual delivery dates are associated with specific order statuses to determine which orders can be used for delay prediction.

In [7]:
pd.crosstab(
    orders['order_status'],
    orders['order_delivered_customer_date'].isna(),
    rownames=['Order Status'],
    colnames=['Delivery Date Missing']
)

Delivery Date Missing,False,True
Order Status,,
approved,0,2
canceled,6,619
created,0,5
delivered,96470,8
invoiced,0,314
processing,0,301
shipped,0,1107
unavailable,0,609


## Part 2: Data Cleaning

### 8. Filter Valid Delivered Orders

Keep only delivered orders with non-missing actual delivery timestamps to ensure that the delivery delay label can be determined.

In [8]:
orders_clean = orders[
    (orders['order_status'] == 'delivered') &
    (orders['order_delivered_customer_date'].notna())
].copy()

print("Original orders:", len(orders))
print("Retained orders:", len(orders_clean))
print("Excluded orders:", len(orders) - len(orders_clean))

Original orders: 99441
Retained orders: 96470
Excluded orders: 2971


### 9. Convert Timestamp Columns

Convert timestamp columns to datetime format for subsequent time-based feature construction and validation.

In [9]:
timestamp_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in timestamp_cols:
    orders_clean[col] = pd.to_datetime(orders_clean[col], errors='coerce')

print("Timestamp data types:")
display(orders_clean[timestamp_cols].dtypes.to_frame('dtype'))

print("\nMissing values after conversion:")
display(orders_clean[timestamp_cols].isna().sum().to_frame('missing_count'))

Timestamp data types:


,dtype
order_purchase_timestamp,datetime64[ns]
order_approved_at,datetime64[ns]
order_delivered_carrier_date,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]



Missing values after conversion:


,missing_count
order_purchase_timestamp,0
order_approved_at,14
order_delivered_carrier_date,1
order_delivered_customer_date,0
order_estimated_delivery_date,0


### 10. Handle Missing and Invalid Values

Identify missing timestamps and check for chronological inconsistencies before deciding which records to exclude.

In [10]:
timestamp_checks = {
    'Missing approval timestamp':
        orders_clean['order_approved_at'].isna().sum(),

    'Missing carrier timestamp':
        orders_clean['order_delivered_carrier_date'].isna().sum(),

    'Approval before purchase':
        (orders_clean['order_approved_at'] <
         orders_clean['order_purchase_timestamp']).sum(),

    'Delivery before purchase':
        (orders_clean['order_delivered_customer_date'] <
         orders_clean['order_purchase_timestamp']).sum(),

    'Delivery before approval':
        (orders_clean['order_delivered_customer_date'] <
         orders_clean['order_approved_at']).sum(),

    'Estimated delivery before purchase':
        (orders_clean['order_estimated_delivery_date'] <
         orders_clean['order_purchase_timestamp']).sum()
}

display(pd.Series(timestamp_checks).to_frame('count'))

,count
Missing approval timestamp,14
Missing carrier timestamp,1
Approval before purchase,0
Delivery before purchase,0
Delivery before approval,61
Estimated delivery before purchase,0


#### Apply Timestamp Cleaning Rules

Exclude orders with missing approval timestamps or delivery timestamps earlier than approval. Retain orders with missing carrier timestamps, as carrier information is not required for prediction.

In [11]:
# Keep orders with valid approval timestamps and chronological order
valid_orders = (
    orders_clean['order_approved_at'].notna() &
    (orders_clean['order_delivered_customer_date'] >=
     orders_clean['order_approved_at'])
)

original_count = len(orders_clean)

orders_clean = orders_clean.loc[valid_orders].copy()

print("Orders before cleaning:", original_count)
print("Orders after cleaning:", len(orders_clean))
print("Excluded orders:", original_count - len(orders_clean))

Orders before cleaning: 96470
Orders after cleaning: 96395
Excluded orders: 75


## Part 3: Data Integration

### 11. Aggregate Order Items

Aggregate item-level records by `order_id` to construct order-level features, including item count, total price, total freight value, and distinct seller count.

In [12]:
items_agg = (
    items.groupby('order_id', as_index=False)
    .agg(
        item_count=('order_item_id', 'count'),
        total_price=('price', 'sum'),
        total_freight=('freight_value', 'sum'),
        seller_count=('seller_id', 'nunique')
    )
)

print("Original item rows:", len(items))
print("Orders after aggregation:", len(items_agg))
print("Duplicate order_id:", items_agg['order_id'].duplicated().sum())

display(items_agg.head())

Original item rows: 112650
Orders after aggregation: 98666
Duplicate order_id: 0


,order_id,item_count,total_price,total_freight,seller_count
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,1
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,1
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,1
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,1


### 12. Merge Orders and Order Items

Merge the cleaned orders with aggregated item-level features using `order_id`. Use a left join to preserve all cleaned orders and check for unmatched records.

In [13]:
orders_merged = orders_clean.merge(
    items_agg,
    on='order_id',
    how='left',
    validate='one_to_one',
    indicator=True
)

print("Orders before merge:", len(orders_clean))
print("Orders after merge:", len(orders_merged))

print("\nMerge results:")
display(orders_merged['_merge'].value_counts().to_frame('count'))

print("\nMerged dataset preview:")
display(orders_merged.head())

Orders before merge: 96395
Orders after merge: 96395

Merge results:


,count
_merge,
both,96395
left_only,0
right_only,0



Merged dataset preview:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,item_count,total_price,total_freight,seller_count,_merge
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,29.99,8.72,1,both
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1,118.70,22.76,1,both
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1,159.90,19.22,1,both
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1,45.00,27.20,1,both
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1,19.90,8.72,1,both
